## Kakao kanana-nano-2.1b-instruct 모델을 사용한 영/한 번역기

from transformers import pipeline
import torch
import re

In [1]:
from transformers import pipeline
import torch
import re

In [11]:
# 1) 사용할 모델 이름
model_name = "kakaocorp/kanana-nano-2.1b-instruct"

In [12]:
# 2) pipeline 생성 (한 번만)
translator = pipeline(
    "text-generation",
    model=model_name,
    tokenizer=model_name,
    trust_remote_code=True,
    # device_map="auto",
    device=0,
    torch_dtype=torch.bfloat16
)

Device set to use cuda:0


In [ ]:
def clean_translation(full_text: str) -> str:
    """
    1. 줄바꿈 전까지 취하고
    2. 문장 종결부호(.!? 등)까지 잘라내어
    3. 나머지는 모두 버립니다.
    """
    first_line = full_text.split("\n")[0].strip()
    
    # 첫 번째 줄에서 문장 종료 기호(.!? 중 하나)가 처음 나올 때까지의 문장 전체를 캡처
    m = re.match(r'(.+?[\.!?])', first_line)
    
    # first_line에서 마침표/느낌표/물음표로 끝나는 문장만 잘라내고,그런 문장이 없으면 전체 줄을 그대로 사용
    return m.group(1).strip() if m else first_line

In [14]:
print("=== 영어/ 한국어 번역기 ===")
print("종료하려면 'quit' 을 입력하세요.\n")

while True:
    src = input("영어 문장 입력: ").strip()
    if src.lower() == "quit":
        print("번역을 종료합니다.")
        break

    prompt = (
        "영어를 한국어로 번역해줘:\n"
        f"English: {src}\n"
        "Korean:"
    )

    outputs = translator(
        prompt,
        max_new_tokens=50,
        eos_token_id=translator.tokenizer.eos_token_id,
        pad_token_id=translator.tokenizer.pad_token_id,
        return_full_text=False
    )
    full = outputs[0]["generated_text"].strip()
    print(clean_translation(full), "\n")

=== 영어/ 한국어 번역기 ===
종료하려면 'quit' 을 입력하세요.

first_line 나는 소년입니다.
나는 소년입니다. 

번역을 종료합니다.


### 출력 cleaning이 없는 코드

In [ ]:
def translate_en_to_ko(text: str, max_tokens: int = 50) -> str:
    """
    영어 문장 text를 한국어로 번역해서 반환합니다.
    """
    prompt = (
        "영어를 한국어로 번역해줘:\n"
        f"English: {text}\n"
        "Korean:"
    )
    outputs = translator(
        prompt,
        max_new_tokens=max_tokens,
        eos_token_id=translator.tokenizer.eos_token_id,
        pad_token_id=translator.tokenizer.pad_token_id,
        return_full_text=False
    )
    full = outputs[0]["generated_text"].strip()
    # 첫 번째 줄까지만 추출
    return full.split("\n")[0]

# ——— 여기서부터 바로 실행 ———

print("=== 번역 테스트 시작 ===")
examples = [
    "I am a boy.",
    "This model is really amazing!",
    "How are you today?"
]

for src in examples:
    result = translate_en_to_ko(src)
    print(f"{src}  →  {result}")


=== 번역 테스트 시작 ===
I am a boy.  →  저는 소년입니다. (저는 남자 아이입니다.) 
This model is really amazing!  →  이 모델은 정말 대단해! 
How are you today?  →  오늘 기분이 어떠세요? 
